# Combined Amber RAG (ChromaDB archive + PDF FAISS) — ChatGPT version

This is the **ChatGPT** variant of the combined Amber RAG notebook. Everything in the retrieval pipeline (Chroma archive + FAISS PDF, embeddings, hybrid retriever, prompt) is identical to the Ollama version so the generated answers can be compared apples-to-apples.

**Only the LLM changes:** `ChatOllama("llama3.1:8b")` → `ChatOpenAI("gpt-4o-mini")`.

- **Store A** — persistent **Chroma** at `/opt/chromadb/data/prompt_db`.
- **Store B** — in-memory **FAISS** built once from `Amber25.pdf`, cached to `./faiss_pdf_index/` so it loads instantly on subsequent runs.

Both stores use `sentence-transformers/all-MiniLM-L6-v2` and L2 (squared) distance, so scores are directly comparable across stores.

## 1) Setup

In [22]:
import os
from typing import List, Tuple

import chromadb
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI

### OpenAI API key

This notebook calls the OpenAI Chat Completions API via `langchain-openai`. Before launching Jupyter, export your key:

```bash
export OPENAI_API_KEY="sk-..."
```

If `langchain-openai` is not installed yet:

```bash
pip install -U langchain-openai
```

In [23]:
from dotenv import load_dotenv
load_dotenv()  # reads .env from the current folder into os.environ

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to your .env file."
    )
print("OPENAI_API_KEY detected (length={}).".format(len(os.environ["OPENAI_API_KEY"])))

OPENAI_API_KEY detected (length=164).


## 2) Shared embedding model
Both stores must use the same embedding model for the scores to be comparable.

In [24]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## 3) Open both stores

- **Archive store** — persistent Chroma at `/opt/chromadb/data/prompt_db`.
- **PDF store** — in-memory FAISS. On first run, it parses `Amber25.pdf`, embeds the chunks, and saves a FAISS index to `./faiss_pdf_index/`. On every subsequent run, it loads that cached index in < 1s instead of re-embedding ~3,800 chunks.

Delete `./faiss_pdf_index/` to force a rebuild (e.g. when the PDF or chunking settings change).

In [25]:
# --- Store A: archive / emails (persistent Chroma) ---
ARCHIVE_DB_PATH = "/opt/chromadb/data/prompt_db"
archive_client = chromadb.PersistentClient(path=ARCHIVE_DB_PATH)
vectorstore_archive = Chroma(
    client=archive_client,
    collection_name="amber_messages",
    embedding_function=embeddings,
)

# --- Store B: PDF (in-memory FAISS with cached index on disk) ---
PDF_PATH = "Amber25.pdf"
FAISS_INDEX_DIR = "./faiss_pdf_index"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100


def _clean_pdf_text(text: str) -> str:
    text = " ".join(text.split())
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    return text


def _build_pdf_chunks(pdf_path: str) -> List[Document]:
    pages = PyPDFLoader(pdf_path).load()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=[" "]
    )
    chunks: List[Document] = []
    for page_num, page in enumerate(pages):
        cleaned = _clean_pdf_text(page.page_content)
        if len(cleaned.strip()) < 50:
            continue
        chunks.extend(splitter.create_documents(
            texts=[cleaned],
            metadatas=[{
                **page.metadata,
                "page": page_num + 1,
                "total_pages": len(pages),
                "chunk_method": "smart_pdf_processor",
                "char_count": len(cleaned),
            }],
        ))
    return chunks


if os.path.isdir(FAISS_INDEX_DIR):
    vectorstore_pdf = FAISS.load_local(
        FAISS_INDEX_DIR, embeddings, allow_dangerous_deserialization=True
    )
    print(f"Loaded cached FAISS index from {FAISS_INDEX_DIR}")
else:
    print(f"No cached index at {FAISS_INDEX_DIR} — building from {PDF_PATH} ...")
    pdf_chunks = _build_pdf_chunks(PDF_PATH)
    print(f"  {len(pdf_chunks)} chunks, embedding ...")
    vectorstore_pdf = FAISS.from_documents(pdf_chunks, embeddings)
    vectorstore_pdf.save_local(FAISS_INDEX_DIR)
    print(f"  Saved FAISS index to {FAISS_INDEX_DIR}")

print(f"Archive store: {vectorstore_archive._collection.count()} vectors @ {ARCHIVE_DB_PATH}")
print(f"PDF store:     {vectorstore_pdf.index.ntotal} vectors (FAISS in-memory)")

Loaded cached FAISS index from ./faiss_pdf_index
Archive store: 42935 vectors @ /opt/chromadb/data/prompt_db
PDF store:     3793 vectors (FAISS in-memory)


## 4) Hybrid retriever — merge both stores, keep top 5
Each store is queried for its own top-K. We tag every doc with which store it came from, sort by raw L2 distance (lower = closer), and keep the best `k_total` overall.

In [26]:
def hybrid_retrieve(
    query: str,
    k_total: int = 5,
    k_archive: int = 10,
    k_pdf: int = 10,
) -> List[Document]:
    """Query both stores, merge by distance (lower = closer), return top k_total docs."""
    scored: List[Tuple[Document, float]] = []

    for doc, score in vectorstore_archive.similarity_search_with_score(query, k=k_archive):
        doc.metadata = {**doc.metadata, "store": "archive", "score": float(score)}
        scored.append((doc, score))

    for doc, score in vectorstore_pdf.similarity_search_with_score(query, k=k_pdf):
        doc.metadata = {**doc.metadata, "store": "pdf", "score": float(score)}
        scored.append((doc, score))

    scored.sort(key=lambda pair: pair[1])  # L2: lower is closer
    return [doc for doc, _ in scored[:k_total]]

# Quick smoke test
hits = hybrid_retrieve("What is Amber?", k_total=5)
for i, d in enumerate(hits, 1):
    print(f"{i}. [{d.metadata.get('store')}] score={d.metadata.get('score'):.4f} "
          f"page={d.metadata.get('page', '-')}")
    print("   ", d.page_content[:160].replace("\n", " "), "...")


1. [archive] score=0.3206 page=-
    Steve Seibold < seibold.chemistry.msu.edu > (Wed, 5 May 2010 08:38:46 -0400): Hi Sorry if this is an ignorant question, but I have been reading on the Amber ema ...
2. [archive] score=0.3282 page=-
    via AMBER < amber.ambermd.org > (Mon, 12 Jun 2023 02:03:48 +0800 (CST)):  ...
3. [archive] score=0.3356 page=-
    < steinbrt.rci.rutgers.edu > (Thu, 28 Mar 2013 09:16:56 -0400 (EDT)): Hi,  > Thank you very much.. I am very new to amber. First time i am using&nbsp;  > follow ...
4. [archive] score=0.3400 page=-
    via AMBER < amber.ambermd.org > (Mon, 1 Jan 2024 00:25:24 +0800 (CST)):  ...
5. [archive] score=0.3631 page=-
    Carlos Simmerling < carlos.csb.sunysb.edu > (Fri, 15 Sep 2006 15:47:34 -0400): I apologize for using the Amber list to advertise my own article, but I think it  ...


## 5) LLM + prompt
Using OpenAI's `gpt-4o-mini` via `langchain-openai`. Temperature is 0 so answers are as deterministic as the API allows, which makes side-by-side comparison with the Ollama run fairer.

In [27]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [28]:
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).

CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention any Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output the final answer.
For source citations, include the source page number (for PDF) or source label (for archive) at the beginning.

Context:
{context}

Question: {question}

Answer:""")

## 6) LCEL chain using the hybrid retriever

In [29]:
def format_docs(docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "unknown")
        source = doc.metadata.get("source", "unknown_source")
        page = doc.metadata.get("page", "-")
        score = doc.metadata.get("score", None)
        score_str = f" | Score: {score:.4f}" if isinstance(score, (int, float)) else ""
        formatted.append(
            f"[Chunk {i} | Store: {store} | Source: {source} | Page: {page}{score_str}]\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)

In [30]:
# Wrap the hybrid retrieval in a RunnableLambda so it plugs into LCEL like any other retriever.
hybrid_retriever = RunnableLambda(lambda q: hybrid_retrieve(q, k_total=5))

rag_chain_lcel = (
    {
        "context": hybrid_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

## 7) Query helper — answer + the top-5 sources that fed it

In [31]:
def query_rag(question: str, k_total: int = 5):
    print(f"Question: {question}")
    print("-" * 60)

    docs = hybrid_retrieve(question, k_total=k_total)
    answer = rag_chain_lcel.invoke(question)

    print("Answer:")
    print(answer)

    print("\nTop sources used:")
    for i, doc in enumerate(docs, 1):
        store = doc.metadata.get("store", "?")
        page = doc.metadata.get("page", "-")
        src = doc.metadata.get("source", "-")
        score = doc.metadata.get("score")
        score_str = f"{score:.4f}" if isinstance(score, (int, float)) else "-"
        print(f"\n--- Source {i} [{store}] page={page} src={src} score={score_str} ---")
        print(doc.page_content[:240].replace("\n", " "), "...")

    return answer, docs

## 8) Try it

In [32]:
_ = query_rag("What is Amber?")

Question: What is Amber?
------------------------------------------------------------
Answer:
Amber is a suite of molecular simulation programs that includes tools for molecular dynamics, energy minimization, and other computational chemistry tasks. It is widely used for simulating biomolecules such as proteins and nucleic acids.

Amber consists of several components, including the AMBER molecular dynamics package and AmberTools, which provides additional functionalities and tools for preparing and analyzing molecular simulations. The suite is designed to facilitate the study of molecular interactions and dynamics, making it a valuable resource for researchers in computational chemistry and biophysics. 

For more detailed information, users can refer to the Amber website and its associated documentation, which includes tutorials and manuals to help new users get started with the software.

Top sources used:

--- Source 1 [archive] page=- src=- score=0.3206 ---
Steve Seibold < seibold.c

In [33]:
_ = query_rag("Why should SHAKE be disabled during minimization in AMBER?")

Question: Why should SHAKE be disabled during minimization in AMBER?
------------------------------------------------------------
Answer:
[Chunk 1 | Store: archive | Source: unknown_source | Page: - | Score: 0.2841]  
SHAKE should be disabled during minimization in AMBER because the minimizer is not aware of the constraints imposed by the SHAKE algorithm. This can lead to issues where the minimization process does not effectively resolve bad contacts or optimize the structure, as the minimizer cannot account for the dynamics of the SHAKE constraints. 

The primary reason is that SHAKE is an algorithm based on dynamics, and during minimization, the algorithm's constraints can interfere with the minimization process. As a result, it is generally recommended to perform minimizations without SHAKE to ensure that the minimizer can fully optimize the structure without being limited by the constraints imposed by SHAKE. 

In cases where SHAKE must be used, it is suggested to perform only short

In [34]:
_ = query_rag("How do I obtain a Z-DNA structure from NAB?")

Question: How do I obtain a Z-DNA structure from NAB?
------------------------------------------------------------
Answer:
[Chunk 1 | Store: archive | Source: unknown_source | Page: - | Score: 0.2341]  
To obtain a Z-DNA structure from NAB, you can use the website http://w3dna.rutgers.edu, which provides tools for generating various DNA structures, including Z-DNA. While NAB itself can create A-DNA structures (as noted in section 16.14 of the AmberTools manual), it is not specifically designed for Z-DNA. 

For practical guidance, you may also consider using the make -na-server tool mentioned in the discussions, which can help in generating A and B forms of DNA, and may provide insights or options for Z-DNA as well. 

In summary, for Z-DNA, the recommended approach is to utilize the resources available at w3dna.rutgers.edu.

Top sources used:

--- Source 1 [archive] page=- src=- score=0.2341 ---
arnab bhattacharya < arnab.chemistry.gmail.com > (Tue, 18 Jun 2013 16:34:31 +0300): Hi,  Won

In [35]:
_ = query_rag("How can I get SHAKE to consider two different residue names to be water?")

Question: How can I get SHAKE to consider two different residue names to be water?
------------------------------------------------------------
Answer:
[Chunk 1 | Store: archive | Source: unknown_source | Page: - | Score: 0.3622]

To get SHAKE to consider two different residue names as water, you can rename the water residues in your topology file. By default, sander/pmemd identifies water residues by the name "WAT." If you want to use different names, such as "CRY" for crystal water and "WAT" for water from the water box, you will need to create a separate topology file that includes these altered residue names. 

This approach allows you to track the different types of water molecules during your simulations while ensuring that they are treated as water by the simulation software. However, it is important to note that the analysis of these residues will depend on how your analysis software is set up to recognize these names. If your software cannot accommodate different residue names

In [36]:
_ = query_rag("Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?")

Question: Best OS recommendation for installing Amber + CUDA on an RTX 3070 server: Rocky Linux vs Ubuntu? What setup is known to work?
------------------------------------------------------------
Answer:
[Chunk 1 | Source: unknown_source | Page: -]

The recommended operating system for installing Amber and CUDA on an RTX 3070 server is Ubuntu 22.04 LTS. This setup is known to work well with AmberTools23 and CUDA 12.3.

The user experience shared indicates that while Rocky Linux 8.9 can launch Schrodinger, issues were encountered with installing pmem.cuda, likely due to compatibility problems with CUDA 11 on that OS. In contrast, Ubuntu 22.04 LTS has been confirmed to work effectively with Amber and the latest CUDA versions. It is advisable to stick with Ubuntu 22.04 LTS because later releases may come with a higher-than-supported GCC compiler, which could lead to additional complications during installation. 

For practical guidance, if you decide to switch to Ubuntu, be prepared to m

In [37]:
_ = query_rag("How do I use paramfit to generate force field parameters for boron-containing compounds?")

Question: How do I use paramfit to generate force field parameters for boron-containing compounds?
------------------------------------------------------------
Answer:
[Chunk 1 | Store: archive | Source: unknown_source | Page: - | Score: 0.2203]

To generate force field parameters for boron-containing compounds using the paramfit program, you should first consult the AMBER archive for relevant discussions and guidance. A search for "boron" on the ambermd.org website can yield useful posts and insights from the community.

The process typically involves using quantum mechanical calculations to derive the necessary parameters, as there are no predefined force field parameters for boron in the standard AMBER force fields. You may need to manually fit parameters based on your specific compound's characteristics and the results of your quantum calculations.

Additionally, consider exploring other tools like mgdx or looking for existing literature that may have addressed similar compounds, a

In [38]:
_ = query_rag("I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?")

Question: I have run a short TI simulation on a system using pmemd wherein, I have in one state bonded disulphide bridge (state A) and in the other unbonded bridge (State B). The peptide I am working with is 40 residues long. Can somebody kindly suggest me how do I extract pdb of the two states?
------------------------------------------------------------
Answer:
[Chunk 1 | Store: archive | Source: unknown_source | Page: - | Score: 0.2182]

To extract PDB files for the two states of your system (bonded disulfide bridge in state A and unbonded in state B), you can use cpptraj to manipulate the trajectory data. Here’s a suggested approach:

1. **Extract the trajectory**: Use cpptraj to read your trajectory file and specify the residues you want to keep for each state.
2. **Adjust the strip mask**: For state A, you can keep the residues involved in the bonded state, and for state B, you can strip those residues and keep the others.

Here’s a basic outline of the cpptraj commands you might

In [39]:
_ = query_rag("""
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?
""")

Question: 
I'm using CPPTRAJ to analyze simulations of parallel strand DNA and want to
> get step parameters via nastruct. Does anyone know how nastruct identify
> the molecule as parallel strands? If I write "guessbp bptype para" in the
> nastruct command, the program will simply get stuck and look as if it is
> not continuing to run at all. If I just write "guessbp", it would seem to
> treat the molecule as anti-parallel. What is the correct way to do that?

------------------------------------------------------------
Answer:
[Chunk 1 | Store: archive | Source: unknown_source | Page: - | Score: 0.1265]

To analyze parallel strand DNA using CPPTRAJ's nastruct command, you should ensure that you are using a version of cpptraj that is 6.18.0 or later, as base pair detection has been significantly improved in these versions. If you are using an older version, it is recommended to upgrade.

The nastruct command does not currently support explicitly defining the base pairing type for paral